# FDT grayscale skeleton for dendrite masks

This notebook tests a fuzzy-distance-transform (FDT) style dendrite mask on a
small patch subset. It is a **visual sanity check** on ~8 structural patches,
not a full pseudo-label generation run.

## Method

For the structural channel `I`, compute a per-patch fuzzy membership
`mu(x) = clip((I(x) - t_low) / (t_high - t_low), 0, 1)` with percentile
defaults `(t_low, t_high) = (p50, p95)`. Build the binary support `mu > 0`,
compute a cheap depth surrogate `pseudo_fdt = EDT(mu > 0) * mu`, optionally
smooth it with `sigma = 1`, skeletonize the support with Zhang-Suen
(`skimage.morphology.skeletonize`), prune short leaf branches, and dilate the
pruned skeleton by a small disk.

## Caveat: this is not Saha's exact FDT skeleton

Saha's FDT is a path-based fuzzy metric on grayscale objects, and a true
grayscale FDT skeleton needs FDT-aware thinning or ordered deletion. This
notebook does **not** implement that. The `pseudo_fdt` map here is only a cheap
surrogate for "depth inside a fuzzy bright set". The skeleton is computed on
`mu > 0`, not by grayscale FDT-ordered thinning.

## Why try this on these MIPs?

Ridge filters ask whether a neighbourhood already looks line-like. That can
fail when a dendrite appears as bright puncta plus dim bridges. FDT-style depth
is more forgiving: puncta have finite depth, dim inter-punctum signal has small
but non-zero depth, and a skeleton can still thread through both without an
early binary "is this a line" decision. This is the same intuition behind the
grayscale FDT mode (`2dSpAn-Auto.f`) in 2dSpAn-Auto.

## Context / provenance

- Saha, Wehrli, Gomberg, **"Fuzzy distance transform: theory, algorithms, and
  applications"**, *Computer Vision and Image Understanding* 86(3):171-190,
  2002.
- Saha, Wehrli, **"Measurement of trabecular bone thickness in the limited
  resolution regime of in vivo MRI by fuzzy distance transform"**, *IEEE
  Transactions on Medical Imaging* 23(1):53-62, 2004.
- Bhattacharya et al., **"2dSpAn-Auto: an automated tool for analysis of
  two-dimensional dendritic spine images"**, *BMC Bioinformatics* 26, 2025
  (PMC12211165). Their grayscale FDT dendrite mode is the direct precedent for
  this quick microscopy sanity check.

## Imports + repo root

In [ ]:
import ast
import csv
import importlib.util
import io
import sys
import tarfile
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import distance_transform_edt, gaussian_filter
from skimage.morphology import dilation, disk, skeletonize

plt.rcParams["figure.dpi"] = 120

In [ ]:
NB_DIR = Path.cwd().resolve()
REPO_ROOT = NB_DIR
while REPO_ROOT.parent != REPO_ROOT and not (REPO_ROOT / ".git").is_dir():
    REPO_ROOT = REPO_ROOT.parent

SRC_ROOT = REPO_ROOT / "src"
for p in (SRC_ROOT, REPO_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print("REPO_ROOT:", REPO_ROOT)
print("SRC_ROOT :", SRC_ROOT)

from synaptic_ssl.pseudolabels.blobs import _prune_skeleton_branches

def load_reassemble_helpers():
    reassemble_path = SRC_ROOT / "synaptic_ssl" / "utils_data" / "reassemble.py"
    spec = importlib.util.spec_from_file_location("reassemble_standalone", reassemble_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.ImageCache, module.load_patch_records

ImageCache, load_patch_records = load_reassemble_helpers()

def load_sanity_train_indices():
    try:
        from synaptic_ssl.training.sanity_batch import SANITY_TRAIN_INDICES
        return tuple(int(i) for i in SANITY_TRAIN_INDICES), False
    except Exception as exc:
        src = (SRC_ROOT / "synaptic_ssl" / "training" / "sanity_batch.py").read_text(encoding="utf-8")
        tree = ast.parse(src)
        for node in tree.body:
            value = None
            if isinstance(node, ast.Assign):
                names = [t.id for t in node.targets if isinstance(t, ast.Name)]
                if "SANITY_TRAIN_INDICES" in names:
                    value = node.value
            elif isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name):
                if node.target.id == "SANITY_TRAIN_INDICES":
                    value = node.value
            if value is not None:
                return tuple(int(i) for i in ast.literal_eval(value)), True
        raise RuntimeError("SANITY_TRAIN_INDICES not found in sanity_batch.py") from exc

SANITY_TRAIN_INDICES, _SANITY_AST_FALLBACK = load_sanity_train_indices()

try:
    import pyfdt  # type: ignore  # noqa: F401
    _HAS_PYFDT = True
except Exception:
    _HAS_PYFDT = False

print("SANITY_TRAIN_INDICES:", SANITY_TRAIN_INDICES)
print("sanity_batch AST fallback:", _SANITY_AST_FALLBACK)
print("pyfdt available:", _HAS_PYFDT)

## Configuration

`PATCH_ROOT` is the canonical dataset location. If it is missing locally but
`data/patches.tar.gz` exists, the notebook extracts a tiny 8-patch demo subset
into `data/patches_128_demo/` so the visual check still runs.

In [ ]:
PATCH_ROOT = REPO_ROOT / "data" / "patches_128"
PATCH_ARCHIVE = REPO_ROOT / "data" / "patches.tar.gz"
DEMO_PATCH_ROOT = REPO_ROOT / "data" / "patches_128_demo"
EXCLUDE_PATTERNS = ["KONTROLA"]

STRUCTURAL_CHANNEL = 2
N_DEMO_PATCHES = 8

PERCENTILE_GRID = [(40, 90), (50, 95), (60, 98)]
PRUNE_GRID = (10, 20)
SIGMA = 1.0
DILATE_R = 2

PARAM_GRID = [
    dict(
        t_low_pct=t_low_pct,
        t_high_pct=t_high_pct,
        sigma=SIGMA,
        prune_len=prune_len,
        dilate_r=DILATE_R,
    )
    for (t_low_pct, t_high_pct), prune_len in product(PERCENTILE_GRID, PRUNE_GRID)
]
RECOMMENDED_PARAMS = dict(
    t_low_pct=50,
    t_high_pct=95,
    sigma=1.0,
    prune_len=10,
    dilate_r=2,
)

print("PATCH_ROOT      :", PATCH_ROOT)
print("PATCH_ARCHIVE   :", PATCH_ARCHIVE)
print("N_DEMO_PATCHES  :", N_DEMO_PATCHES)
print("PARAM_GRID SIZE :", len(PARAM_GRID))

## Load patches via `ImageCache`

In [ ]:
def resolve_demo_indices(n_records, base_indices, n_target=8):
    if n_records <= 0:
        return []

    target = min(int(n_target), int(n_records))
    resolved, seen = [], set()
    for idx in base_indices:
        idx = min(max(int(idx), 0), n_records - 1)
        if idx not in seen:
            resolved.append(idx)
            seen.add(idx)
        if len(resolved) == target:
            return resolved

    cursor = resolved[-1] + 1 if resolved else 0
    while len(resolved) < target and cursor < n_records:
        if cursor not in seen:
            resolved.append(cursor)
            seen.add(cursor)
        cursor += 1

    cursor = 0
    while len(resolved) < target and cursor < n_records:
        if cursor not in seen:
            resolved.append(cursor)
            seen.add(cursor)
        cursor += 1

    return resolved


def _read_archive_records(archive_path):
    with tarfile.open(archive_path, "r:gz") as tf:
        with tf.extractfile("data/patches_128/index.csv") as raw:
            text = io.TextIOWrapper(raw, encoding="utf-8")
            rows = list(csv.DictReader(text))
    return rows


def _extract_demo_subset(archive_path, out_root, exclude_patterns, base_indices, n_target):
    rows = _read_archive_records(archive_path)
    pats = [p.upper() for p in exclude_patterns]
    filtered = [r for r in rows if not any(p in r["source_image"].upper() for p in pats)]
    demo_indices = resolve_demo_indices(len(filtered), base_indices, n_target=n_target)
    subset = [filtered[i] for i in demo_indices]

    out_root.mkdir(parents=True, exist_ok=True)
    wanted = {row["filename"] for row in subset}
    for stale in out_root.glob("*.npy"):
        if stale.name not in wanted:
            stale.unlink()

    with tarfile.open(archive_path, "r:gz") as tf:
        for row in subset:
            dst = out_root / row["filename"]
            if dst.exists():
                continue
            member = f"data/patches_128/{row['filename']}"
            with tf.extractfile(member) as src, open(dst, "wb") as fh:
                fh.write(src.read())

    with open(out_root / "index.csv", "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(subset[0].keys()))
        writer.writeheader()
        writer.writerows(subset)

    return out_root


if PATCH_ROOT.exists():
    ACTIVE_PATCH_ROOT = PATCH_ROOT
    patch_records = load_patch_records(ACTIVE_PATCH_ROOT, EXCLUDE_PATTERNS)
    demo_positions = resolve_demo_indices(
        len(patch_records),
        SANITY_TRAIN_INDICES,
        n_target=N_DEMO_PATCHES,
    )
else:
    if not PATCH_ARCHIVE.exists():
        raise FileNotFoundError(f"Neither {PATCH_ROOT} nor {PATCH_ARCHIVE} exists.")
    ACTIVE_PATCH_ROOT = _extract_demo_subset(
        PATCH_ARCHIVE,
        DEMO_PATCH_ROOT,
        EXCLUDE_PATTERNS,
        SANITY_TRAIN_INDICES,
        N_DEMO_PATCHES,
    )
    patch_records = load_patch_records(ACTIVE_PATCH_ROOT, EXCLUDE_PATTERNS)
    demo_positions = list(range(len(patch_records)))

cache = ImageCache(ACTIVE_PATCH_ROOT, patch_records)
demo_items = []
for pos in demo_positions:
    patch = cache.get_patch(pos).astype(np.float32)
    rec = patch_records[pos]
    demo_items.append(
        dict(
            pos=pos,
            filename=rec["filename"],
            source_image=rec["source_image"],
            image_index=int(rec["image_index"]),
            grid_row=int(rec["grid_row"]),
            grid_col=int(rec["grid_col"]),
            structural=patch[STRUCTURAL_CHANNEL],
        )
    )

print("ACTIVE_PATCH_ROOT:", ACTIVE_PATCH_ROOT)
print("Loaded demo patches:", len(demo_items))
for item in demo_items:
    print(
        f"{item['pos']:>4d}  {item['filename']}  "
        f"img={item['image_index']}  rc=({item['grid_row']},{item['grid_col']})"
    )

## FDT surrogate + mask builder

`pseudo_fdt` is deliberately cheap:

```python
pseudo_fdt = scipy.ndimage.distance_transform_edt(mu > 0) * mu
```

That is **not** Saha's path-based FDT. It only borrows the same intuition:
large values occur deeper inside a fuzzy bright set than on its fringe.

In [ ]:
def fuzzy_membership(image, t_low_pct=50.0, t_high_pct=95.0):
    image = np.asarray(image, dtype=np.float32)
    t_low = float(np.percentile(image, t_low_pct))
    t_high = float(np.percentile(image, t_high_pct))
    scale = max(t_high - t_low, np.finfo(np.float32).eps)
    mu = np.clip((image - t_low) / scale, 0.0, 1.0)
    return mu.astype(np.float32, copy=False), t_low, t_high


def pseudo_fdt(mu, sigma=1.0):
    mu = np.asarray(mu, dtype=np.float32)
    support = mu > 0
    if support.any():
        fdt = distance_transform_edt(support).astype(np.float32) * mu
        if sigma and sigma > 0:
            fdt = gaussian_filter(fdt, sigma=float(sigma))
    else:
        fdt = np.zeros_like(mu, dtype=np.float32)
    return fdt.astype(np.float32, copy=False)


def fdt_dendrite_mask(
    image,
    t_low_pct=50.0,
    t_high_pct=95.0,
    sigma=1.0,
    prune_len=10,
    dilate_r=2,
):
    mu, t_low, t_high = fuzzy_membership(
        image,
        t_low_pct=t_low_pct,
        t_high_pct=t_high_pct,
    )
    support = mu > 0
    fdt = pseudo_fdt(mu, sigma=sigma)
    raw_skeleton = skeletonize(support)
    pruned = _prune_skeleton_branches(raw_skeleton, max_len=int(prune_len))
    if dilate_r > 0 and pruned.any():
        mask = dilation(pruned, disk(int(dilate_r)))
    else:
        mask = pruned.copy()
    return dict(
        image=np.asarray(image, dtype=np.float32),
        mu=mu,
        fdt=fdt,
        support=support,
        raw_skeleton=raw_skeleton.astype(bool, copy=False),
        skeleton=pruned.astype(bool, copy=False),
        mask=mask.astype(bool, copy=False),
        thresholds=(t_low, t_high),
        params=dict(
            t_low_pct=t_low_pct,
            t_high_pct=t_high_pct,
            sigma=sigma,
            prune_len=prune_len,
            dilate_r=dilate_r,
        ),
    )


def normalize_for_display(image):
    image = np.asarray(image, dtype=np.float32)
    lo, hi = np.percentile(image, [1, 99])
    if hi <= lo:
        hi = lo + np.finfo(np.float32).eps
    return np.clip((image - lo) / (hi - lo), 0.0, 1.0)


def mask_overlay(image, mask, skeleton=None):
    base = normalize_for_display(image)
    rgb = np.dstack([base, base, base])
    if mask.any():
        rgb[mask] = 0.45 * rgb[mask] + 0.55 * np.array([1.0, 0.2, 0.2])
    if skeleton is not None and np.any(skeleton):
        rgb[skeleton] = np.array([0.0, 1.0, 1.0])
    return np.clip(rgb, 0.0, 1.0)


def summarize_outputs(outputs):
    fdt_p95 = [
        float(np.percentile(out["fdt"][out["fdt"] > 0], 95))
        if np.any(out["fdt"] > 0) else 0.0
        for out in outputs
    ]
    return dict(
        support_frac=float(np.mean([out["support"].mean() for out in outputs])),
        skeleton_px=float(np.mean([out["skeleton"].sum() for out in outputs])),
        mask_frac=float(np.mean([out["mask"].mean() for out in outputs])),
        fdt_p95=float(np.mean(fdt_p95)),
    )

## Small parameter sweep

In [ ]:
sweep_results = []
for params in PARAM_GRID:
    outputs = [fdt_dendrite_mask(item["structural"], **params) for item in demo_items]
    label = f"p{params['t_low_pct']}-{params['t_high_pct']}_pr{params['prune_len']}"
    metrics = summarize_outputs(outputs)
    sweep_results.append(dict(label=label, params=params, outputs=outputs, **metrics))

print("Parameter sweep (means over demo patches)")
print("label                support   skel_px   mask    fdt_p95")
for row in sweep_results:
    print(
        f"{row['label']:<18}  {row['support_frac']:>7.3f}  "
        f"{row['skeleton_px']:>7.1f}  {row['mask_frac']:>6.3f}  {row['fdt_p95']:>7.3f}"
    )

In [ ]:
preview_rows = min(2, len(demo_items))
fig, axes = plt.subplots(
    preview_rows,
    len(sweep_results),
    figsize=(3.0 * len(sweep_results), 3.0 * preview_rows),
    constrained_layout=True,
)
axes = np.atleast_2d(axes)

for r in range(preview_rows):
    item = demo_items[r]
    for c, row in enumerate(sweep_results):
        out = row["outputs"][r]
        ax = axes[r, c]
        ax.imshow(mask_overlay(item["structural"], out["mask"], out["skeleton"]))
        ax.set_title(row["label"], fontsize=9)
        if c == 0:
            ax.set_ylabel(item["filename"].replace(".npy", ""), fontsize=8)
        ax.axis("off")

fig.suptitle("Parameter sweep preview: overlay only", y=1.02, fontsize=13)
plt.show()

## Eight-patch visual review

In [ ]:
recommended_outputs = [
    fdt_dendrite_mask(item["structural"], **RECOMMENDED_PARAMS)
    for item in demo_items
]
recommended_summary = summarize_outputs(recommended_outputs)
print("Recommended setting:", RECOMMENDED_PARAMS)
print("Summary:", recommended_summary)

In [ ]:
col_titles = [
    "raw structural",
    "membership mu",
    "pseudo-FDT",
    "support",
    "skeleton",
    "dilated mask",
    "overlay",
]

fig, axes = plt.subplots(
    len(demo_items),
    len(col_titles),
    figsize=(18, 2.4 * len(demo_items)),
    constrained_layout=True,
)
axes = np.atleast_2d(axes)

for r, (item, out) in enumerate(zip(demo_items, recommended_outputs)):
    panels = [
        normalize_for_display(item["structural"]),
        out["mu"],
        out["fdt"],
        out["support"],
        out["skeleton"],
        out["mask"],
        mask_overlay(item["structural"], out["mask"], out["skeleton"]),
    ]
    cmaps = ["gray", "viridis", "magma", "gray", "gray", "gray", None]

    for c, (panel, cmap) in enumerate(zip(panels, cmaps)):
        ax = axes[r, c]
        if cmap is None:
            ax.imshow(panel)
        else:
            ax.imshow(panel, cmap=cmap)
        if r == 0:
            ax.set_title(col_titles[c], fontsize=10)
        ax.axis("off")

    t_low, t_high = out["thresholds"]
    axes[r, 0].set_ylabel(
        f"{item['filename'].replace('.npy', '')}\nimg={item['image_index']} rc=({item['grid_row']},{item['grid_col']})",
        fontsize=8,
        rotation=0,
        ha="right",
        va="center",
        labelpad=50,
    )
    axes[r, 0].text(
        0.02,
        0.02,
        f"p{RECOMMENDED_PARAMS['t_low_pct']}/{RECOMMENDED_PARAMS['t_high_pct']}\n{t_low:.3f}-{t_high:.3f}",
        transform=axes[r, 0].transAxes,
        fontsize=7,
        color="white",
        bbox=dict(facecolor="black", alpha=0.5, pad=2),
    )

fig.suptitle("FDT-style dendrite mask demo on 8 structural patches", fontsize=14, y=1.01)
plt.show()

## Recommendation and caveats

- Start with **`(t_low_pct, t_high_pct) = (50, 95)`**, **`sigma = 1`**,
  **`prune_len = 10`**, **`dilate_r = 2`**. On the local 8-patch dry run this
  was the middle ground between the broad `(40, 90)` masks and the stricter
  `(60, 98)` masks.
- In that dry run, percentile choice mattered more than pruning: mean mask
  coverage was about **0.77** for `(40, 90)`, **0.69** for `(50, 95)`, and
  **0.59** for `(60, 98)`, while `prune_len = 10 -> 20` changed mean mask area
  only slightly.
- If bright puncta halos create too many short side branches, raise
  `prune_len` to **20**.
- `(40, 90)` is the friendlier setting when the shaft is very dim, but it can
  overgrow puncta clusters and merge nearby structures.
- `(60, 98)` is the stricter setting and is more likely to break dim bridges.

### Failure modes

- bright soma rims or dense puncta clusters can dominate the support and yield
  a thick or branchy skeleton;
- weak shafts can disappear if `t_low` is too high;
- the binary support step still discards some grayscale nuance before thinning;
- the dilated skeleton is a centerline mask, not a width estimate.

### Honest approximation note

This notebook uses `EDT(mu > 0) * mu` as a cheap depth surrogate. That is not
Saha's path-based fuzzy distance transform, and `skimage.morphology.skeletonize`
on `mu > 0` is not a true grayscale FDT skeleton. Treat it as a fast candidate
generator for visual triage, not as a faithful reproduction of Saha's FDT
skeleton.